# Medicare Outpatient Claims — Denial Risk Analysis

**Goal:** Identify billing patterns in Medicare outpatient claims that would likely trigger payer denials, and quantify the dollars at risk.

## Status
- [x] Data loaded and profiled
- [x] Service lines normalized (3.77M)
- [x] Finding #1: bundling hypothesis tested (not supported)
- [x] NCCI PTP edit file loaded (1.86M code pairs)
- [x] Finding #2: 80053/80048 hard edit confirmed ($5.1M)
- [ ] Full NCCI violation scan across all claims
- [ ] Denial risk scoring
- [ ] Appeal prioritization

## Dataset
- Source: CMS DE-SynPUF (Synthetic Public Use Files), Outpatient Claims, Sample 1
- Rows: 790,790 | Unique claims: 779,815 | Service lines: 3,774,339
- Date range: Dec 2007 – Dec 2010
- Total payments: $224,524,710
- **Contains only PAID claims — no denials are present in this data**

**Rules source:** CMS NCCI Hospital Procedure-to-Procedure (PTP) Edits, 2026 Q3 (v322r0), 1,864,729 code pairs across four files.

## Approach
Because no public dataset contains denied claims, denial risk is identified by applying real published payer billing rules to paid claims to flag bills that would likely have been rejected.

## Finding #1 — Venipuncture / lab panel bundling (hypothesis NOT supported)

**Hypothesis:** Venipuncture (36415) billed on the same claim as a lab panel (80053, 80048, 85610, 80061, 84443) constitutes an NCCI bundling violation, since specimen collection is generally considered included in the lab panel payment.

**Screen result:** 74,997 claims flagged, $17,621,250 (7.8% of total payments). Flagged claims average $230 vs. $284 overall — risk concentrates in low-value, high-volume lab services where manual appeal costs more than recovery.

**Verification against authoritative source:** Loaded the complete CMS NCCI Hospital PTP edit file (1,864,729 pairs). Both codes appear in the edit tables (36415: 20 appearances; 80053: 25), but **not as a pair in either direction**. Hypothesis not supported by PTP edits.

**Revised conclusion:** The venipuncture/lab-panel relationship is governed by Clinical Lab Fee Schedule payment policy rather than PTP code-pair edits. The $17.6M represents a payment-policy risk population, not a PTP violation. Initial reasoning was directionally correct on the economics but sourced from the wrong policy instrument. Notably, 36415 *is* bundled into critical care and neonatal intensive care E/M codes (99291, 99466–99486) and is mutually exclusive with G0471 — confirming the bundling principle applies, but through different code relationships than hypothesized.

*Analysis run: 2026-08-19*

## Finding #2 — Duplicate metabolic panels (NCCI PTP violation, CONFIRMED)

**Rule:** NCCI PTP edit — 80053 (comprehensive metabolic panel) / 80048 (basic metabolic panel), modifier indicator **0** (never allowed together, no modifier override). The basic panel is a subset of the comprehensive panel; billing both duplicates the same tests.

**Result:** 15,729 claims / $5,101,740

Unlike Finding #1, this violation is directly traceable to a published CMS edit with a hard modifier indicator, making it defensible without further verification. Both codes rank in the top 10 most-billed procedures in this dataset (80053: #4, 80048: #9), so the pattern reflects high-volume systematic billing behavior rather than isolated errors. Related hard edits on the same primary code (80069 renal panel, 80076 hepatic panel, both indicator 0) suggest the scan should be generalized across all panel combinations.

*Analysis run: 2026-08-19*

## Finding #3 — Full NCCI hard-edit scan (CONFIRMED)

**Method:** Filtered NCCI PTP edits to modifier indicator 0 (64,711 pairs, no modifier override permitted). Generated all code-pair combinations per claim across edit-relevant service lines (3,558,615 candidate pairs from 383,360 claims), matched bidirectionally against the edit table.

**Result:**
- 38,350 violating code pairs
- **31,693 claims (4.1% of all claims)**
- **$12,674,090 in claims touched** (upper bound — total payment on any claim containing a violation)
- **~$1,222,331 estimated line-level exposure** (allocating claim payment across service lines; median violating claim contains 11 lines)

**Pricing note:** DE-SynPUF reports payment only at claim level, not per service line. Attributing the full claim payment to a violation overstates exposure by roughly 10x, since the median violating claim contains 11 lines of which typically one is affected. Line-level exposure is estimated by dividing claim payment evenly across lines and multiplying by violating pairs per claim. True exposure lies between these bounds; even allocation is an approximation, as lab codes are typically lower-value than the claim average.

**Concentration:** Violations are highly concentrated. The single pair 80048/80053 (basic vs. comprehensive metabolic panel) accounts for 15,729 pairs — 41% of all violations. Metabolic panel overlaps (80053 with 80048, 80076, 80069) account for ~58% combined.

**Departmental clustering:** Remaining violations group into three functional areas — laboratory (85007/85025 CBC overlap), physical therapy (97124/97140, 97001/97002), and radiology (72193/74160, 74160/74170 overlapping CT regions).

**Operational implication:** A single billing-configuration remediation targeting metabolic panel ordering would address the majority of hard-edit exposure. This reframes the deliverable from a $12.7M aggregate figure to a prioritized, department-level work list.

*Analysis run: 2026-08-19*

## Finding #4 — Violations are systemic, not provider-concentrated

Mapped all 31,693 violating claims to billing providers (PRVDR_NUM).

- 6,293 total providers in dataset
- **4,247 providers (67.5%) have at least one hard-edit violation**
- Top 10 providers account for only **6.4%** of violations
- Top 50 providers account for only **17.9%**

**Interpretation:** Violations show no meaningful provider concentration. Combined with Finding #3 (41% of violations from a single code pair), this indicates a systemic coding pattern rather than outlier billing behavior at specific facilities.

**Operational implication:** Provider-level auditing would be an inefficient remediation strategy. The pattern points instead to shared causes — billing software defaults, standing order-set design, or coding convention — warranting intervention at the tooling and guidance level rather than the facility level.

**Limitation:** DE-SynPUF is synthetically generated and provider assignment may be randomized during synthesis, which would artificially suppress concentration. This finding would require validation against real claims data before being treated as a property of actual provider behavior.

*Analysis run: 2026-08-19*

## Limitations and Validity

**What is verifiable:** All 64,711 edits derive from CMS's published NCCI Hospital PTP file and are traceable to specific rows. Counts and joins are reproducible from the notebook.

**What is conditional:** Violation rates hold for this synthetic dataset. DE-SynPUF is statistically generated from real claims; some relationships are preserved and others are not. Rates observed here should not be assumed to match real Medicare claims.

**What is uncertain:**
- Payment values are binned and capped at $3,300, so all dollar figures are approximate.
- Line-level exposure is estimated, not measured (see Finding #3 pricing note).
- Provider distribution (Finding #4) may reflect synthesis artifacts rather than real billing behavior.
- 2026 Q3 edits applied to 2007–2010 claims; historical edit files are not publicly archived.

**Validation not yet performed:** Results have not been benchmarked against CMS's published improper payment rate (CERT program), which would provide external corroboration of whether the observed violation rate is plausible.

## External Validation — CERT Benchmark

CMS measures Medicare FFS improper payments annually through the Comprehensive 
Error Rate Testing (CERT) program.

- **CERT FY2025:** 6.55% improper payment rate, $28.83B (down from 7.66%/$31.70B in FY2024)
- **This analysis:** 4.1% of claims contain at least one modifier-0 PTP violation

**Interpretation:** The observed rate falls plausibly below the CERT benchmark, 
which is the expected relationship. CERT captures the full universe of improper 
payments — coverage errors, medical necessity failures, insufficient documentation, 
and coding errors. This analysis detects only one narrow category: hard PTP 
code-pair conflicts with no modifier override. A result exceeding CERT would 
have indicated a methodological error; a result at a fraction of CERT is consistent 
with detecting a subset of total improper payment causes.

**Caveats on comparability:** CERT is dollar-weighted; this analysis reports 
claim counts. CERT reflects FY2023–24 claims; this dataset covers 2007–2010. 
CERT explicitly notes its rate does not indicate fraud, consistent with this 
analysis's finding of systemic rather than outlier behavior.

Source: CMS CERT, FY2025 Improper Payments Fact Sheet
*Analysis run: 2026-08-19*

## Data Notes
- Claims may span multiple SEGMENT rows when service lines exceed 45. Payment amounts differ per segment and must be summed per CLM_ID to avoid miscounting.
- Payment values appear binned and capped at $3,300, indicating synthetic privacy treatment. Dollar figures should be treated as approximate.
- Diagnosis codes are ICD-9 (retired in 2015). HCPCS procedure codes remain current.
- NCCI edits applied are 2026 Q3; claims are 2007–2010. CMS publishes only the current quarter. Core bundling relationships are stable over time, but a production implementation would version-match edits to claim service dates.
- DE-SynPUF contains no modifier data, so NCCI modifier-indicator-1 edits (allowed with documentation) cannot be fully evaluated. Analysis therefore prioritizes modifier-indicator-0 edits, which permit no override.
- NCCI files contain AMA-licensed CPT codes and are not redistributed in this repo; download separately from CMS.

In [1]:
import pandas as pd
import numpy as np
import os
from itertools import combinations

In [2]:
claims = pd.read_csv(
    "data/DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv",
    low_memory=False
)

In [3]:
claims.shape

(790790, 76)

In [4]:
claims.head()

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00013D2EFD8E45D1,542192281063886,1,20080904.0,20080904.0,2600RA,50.0,0.0,4.824842e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00016F745862898F,542272281166593,1,20090602.0,20090602.0,3901GS,30.0,0.0,2.963420e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00016F745862898F,542282281644416,1,20090623.0,20090623.0,3939PG,30.0,0.0,5.737808e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0001FDD721E223DC,542642281250669,1,20091011.0,20091011.0,3902NU,30.0,0.0,1.233848e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00024B3D2352D2D0,542242281386963,1,20080712.0,20080712.0,5200TV,30.0,0.0,9.688809e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print("Number of bills:", len(claims))
print("Average payment: $", round(claims["CLM_PMT_AMT"].mean(), 2))
print("Biggest payment: $", claims["CLM_PMT_AMT"].max())
print("Total paid out: $", round(claims["CLM_PMT_AMT"].sum(), 2))

Number of bills: 790790
Average payment: $ 283.92
Biggest payment: $ 3300.0
Total paid out: $ 224524710.0


In [6]:
claims.columns.tolist()

['DESYNPUF_ID',
 'CLM_ID',
 'SEGMENT',
 'CLM_FROM_DT',
 'CLM_THRU_DT',
 'PRVDR_NUM',
 'CLM_PMT_AMT',
 'NCH_PRMRY_PYR_CLM_PD_AMT',
 'AT_PHYSN_NPI',
 'OP_PHYSN_NPI',
 'OT_PHYSN_NPI',
 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
 'ICD9_DGNS_CD_1',
 'ICD9_DGNS_CD_2',
 'ICD9_DGNS_CD_3',
 'ICD9_DGNS_CD_4',
 'ICD9_DGNS_CD_5',
 'ICD9_DGNS_CD_6',
 'ICD9_DGNS_CD_7',
 'ICD9_DGNS_CD_8',
 'ICD9_DGNS_CD_9',
 'ICD9_DGNS_CD_10',
 'ICD9_PRCDR_CD_1',
 'ICD9_PRCDR_CD_2',
 'ICD9_PRCDR_CD_3',
 'ICD9_PRCDR_CD_4',
 'ICD9_PRCDR_CD_5',
 'ICD9_PRCDR_CD_6',
 'NCH_BENE_PTB_DDCTBL_AMT',
 'NCH_BENE_PTB_COINSRNC_AMT',
 'ADMTNG_ICD9_DGNS_CD',
 'HCPCS_CD_1',
 'HCPCS_CD_2',
 'HCPCS_CD_3',
 'HCPCS_CD_4',
 'HCPCS_CD_5',
 'HCPCS_CD_6',
 'HCPCS_CD_7',
 'HCPCS_CD_8',
 'HCPCS_CD_9',
 'HCPCS_CD_10',
 'HCPCS_CD_11',
 'HCPCS_CD_12',
 'HCPCS_CD_13',
 'HCPCS_CD_14',
 'HCPCS_CD_15',
 'HCPCS_CD_16',
 'HCPCS_CD_17',
 'HCPCS_CD_18',
 'HCPCS_CD_19',
 'HCPCS_CD_20',
 'HCPCS_CD_21',
 'HCPCS_CD_22',
 'HCPCS_CD_23',
 'HCPCS_CD_24',
 'HCPCS_CD_25',


In [7]:
print(claims["CLM_FROM_DT"].min())
print(claims["CLM_FROM_DT"].max())

20071212.0
20101231.0


In [8]:
claims["HCPCS_CD_1"].value_counts().head(20)

HCPCS_CD_1
36415    161373
99213     31792
99212     23658
80053     22883
A4657     20237
85610     19812
97110     18410
99214     14890
80048     13866
G0202     13462
71020     12244
99211     11512
G0283      5701
97001      5484
77080      5414
87086      5400
90658      5300
88305      5085
A0425      4865
P9604      4541
Name: count, dtype: int64

In [9]:
hcpcs_cols = [c for c in claims.columns if c.startswith("HCPCS_CD_")]

lines = claims.melt(
    id_vars=["CLM_ID", "DESYNPUF_ID", "CLM_FROM_DT", "CLM_PMT_AMT"],
    value_vars=hcpcs_cols,
    var_name="line_number",
    value_name="hcpcs_code"
).dropna(subset=["hcpcs_code"])

print(lines.shape)
lines.head()

(3774339, 6)


,CLM_ID,DESYNPUF_ID,CLM_FROM_DT,CLM_PMT_AMT,line_number,hcpcs_code
0,542192281063886,00013D2EFD8E45D1,20080904.0,50.0,HCPCS_CD_1,85610
1,542272281166593,00016F745862898F,20090602.0,30.0,HCPCS_CD_1,85610
2,542282281644416,00016F745862898F,20090623.0,30.0,HCPCS_CD_1,71101
3,542642281250669,0001FDD721E223DC,20091011.0,30.0,HCPCS_CD_1,36415
4,542242281386963,00024B3D2352D2D0,20080712.0,30.0,HCPCS_CD_1,76872


In [10]:
# Which claims contain a blood draw?
draw_claims = set(lines[lines["hcpcs_code"] == "36415"]["CLM_ID"])

# Which claims contain a common lab panel?
panels = ["80053", "80048", "85610", "80061", "84443"]
panel_claims = set(lines[lines["hcpcs_code"].isin(panels)]["CLM_ID"])

# Claims with BOTH
both = draw_claims & panel_claims

print("Claims with blood draw:", len(draw_claims))
print("Claims with a lab panel:", len(panel_claims))
print("Claims with BOTH (potential bundling denial):", len(both))

Claims with blood draw: 201084
Claims with a lab panel: 214829
Claims with BOTH (potential bundling denial): 74997


In [11]:
flagged = claims[claims["CLM_ID"].isin(both)]

print("Flagged claims:", len(flagged))
print("Dollars on flagged claims: $", flagged["CLM_PMT_AMT"].sum())
print("Average flagged claim: $", round(flagged["CLM_PMT_AMT"].mean(), 2))
print("Share of total dollars:", round(100 * flagged["CLM_PMT_AMT"].sum() / claims["CLM_PMT_AMT"].sum(), 1), "%")

Flagged claims: 76539
Dollars on flagged claims: $ 17621250.0
Average flagged claim: $ 230.23
Share of total dollars: 7.8 %


In [12]:
print("Unique claim IDs:", claims["CLM_ID"].nunique())
print("Total rows:", len(claims))

Unique claim IDs: 779815
Total rows: 790790


In [13]:
dupes = claims[claims["CLM_ID"].duplicated(keep=False)].sort_values("CLM_ID")
dupes[["CLM_ID", "SEGMENT", "CLM_PMT_AMT"]].head(10)

,CLM_ID,SEGMENT,CLM_PMT_AMT
730885,542012280839653,2,300.0
730933,542012280839653,1,2800.0
188303,542012280850900,2,2300.0
188322,542012280850900,1,2800.0
699152,542012280851865,2,300.0
699153,542012280851865,1,60.0
648884,542012280875173,2,2400.0
648918,542012280875173,1,2800.0
272946,542012280876469,2,2900.0
272963,542012280876469,1,2400.0


In [14]:
claim_totals = claims.groupby("CLM_ID")["CLM_PMT_AMT"].sum().reset_index()
print("Unique claims:", len(claim_totals))
print("Total dollars: $", claim_totals["CLM_PMT_AMT"].sum())

Unique claims: 779815
Total dollars: $ 224524710.0


In [15]:
flagged_totals = claim_totals[claim_totals["CLM_ID"].isin(both)]

print("Flagged claims:", len(flagged_totals))
print("Flagged dollars: $", flagged_totals["CLM_PMT_AMT"].sum())
print("Share of dollars:", round(100 * flagged_totals["CLM_PMT_AMT"].sum() / claim_totals["CLM_PMT_AMT"].sum(), 1), "%")

Flagged claims: 74997
Flagged dollars: $ 17621250.0
Share of dollars: 7.8 %


In [16]:
# !pip3 install openpyxl

In [17]:
cols = ["col1", "col2", "pre1996", "effective_date", "deletion_date", "modifier_indicator", "rationale"]

frames = []
for i in [1, 2, 3, 4]:
    folder = f"ccioph-v322r0-f{i}"
    path = os.path.join("data", folder, f"{folder}.xlsx")
    df = pd.read_excel(path, dtype=str, skiprows=5, names=cols)
    frames.append(df)
    print(f"f{i} loaded:", df.shape)

ncci = pd.concat(frames, ignore_index=True)
print("TOTAL EDIT PAIRS:", len(ncci))

f1 loaded: (475091, 7)
f2 loaded: (475181, 7)
f3 loaded: (474870, 7)
f4 loaded: (439587, 7)
TOTAL EDIT PAIRS: 1864729


In [18]:
hit = ncci[(ncci["col1"] == "80053") & (ncci["col2"] == "36415")]
print(hit.to_string())

Empty DataFrame
Columns: [col1, col2, pre1996, effective_date, deletion_date, modifier_indicator, rationale]
Index: []


In [19]:
print("36415 in col1:", (ncci["col1"] == "36415").sum())
print("36415 in col2:", (ncci["col2"] == "36415").sum())
print("80053 in col1:", (ncci["col1"] == "80053").sum())
print("80053 in col2:", (ncci["col2"] == "80053").sum())

36415 in col1: 2
36415 in col2: 18
80053 in col1: 19
80053 in col2: 6


In [20]:
print("=== What 36415 is bundled INTO (36415 gets denied) ===")
print(ncci[ncci["col2"] == "36415"][["col1", "modifier_indicator", "rationale"]].to_string())

print("\n=== What 80053 bundles (80053 is the payer) ===")
print(ncci[ncci["col1"] == "80053"][["col2", "modifier_indicator", "rationale"]].to_string())

=== What 36415 is bundled INTO (36415 gets denied) ===
          col1 modifier_indicator                                    rationale
157449   0232T                  1  CPT Manual or CMS manual coding instruction
215627   0481T                  1  CPT Manual or CMS manual coding instruction
1812286  99291                  1  CPT Manual or CMS manual coding instruction
1825385  99466                  1  CPT Manual or CMS manual coding instruction
1825718  99467                  1  CPT Manual or CMS manual coding instruction
1825875  99468                  1  CPT Manual or CMS manual coding instruction
1826300  99469                  1  CPT Manual or CMS manual coding instruction
1826723  99471                  1  CPT Manual or CMS manual coding instruction
1827142  99472                  1  CPT Manual or CMS manual coding instruction
1827643  99475                  1  CPT Manual or CMS manual coding instruction
1828099  99476                  1  CPT Manual or CMS manual coding instructi

In [21]:
c80053 = set(lines[lines["hcpcs_code"] == "80053"]["CLM_ID"])
c80048 = set(lines[lines["hcpcs_code"] == "80048"]["CLM_ID"])
violation = c80053 & c80048

print("Claims with both 80053 and 80048:", len(violation))

v = claim_totals[claim_totals["CLM_ID"].isin(violation)]
print("Dollars: $", v["CLM_PMT_AMT"].sum())

Claims with both 80053 and 80048: 15729
Dollars: $ 5101740.0


In [22]:
hard = ncci[ncci["modifier_indicator"] == "0"][["col1", "col2", "rationale"]].copy()
print("Hard edits (modifier 0):", len(hard))

Hard edits (modifier 0): 64711


In [23]:
edit_codes = set(hard["col1"]) | set(hard["col2"])
print("Distinct codes in hard edits:", len(edit_codes))

relevant = lines[lines["hcpcs_code"].isin(edit_codes)]
print("Service lines involving edit codes:", len(relevant), "of", len(lines))

Distinct codes in hard edits: 10355
Service lines involving edit codes: 2244208 of 3774339


In [ ]:
from itertools import combinations
import pandas as pd

grouped = relevant.groupby("CLM_ID")["hcpcs_code"].apply(lambda x: sorted(set(x)))
grouped = grouped[grouped.apply(len) > 1]
print("Claims with 2+ edit-relevant codes:", len(grouped))

pair_rows = []
for clm, codes in grouped.items():
    for a, b in combinations(codes, 2):
        pair_rows.append((clm, a, b))

pairs = pd.DataFrame(pair_rows, columns=["CLM_ID", "code_a", "code_b"])
print("Total candidate pairs:", len(pairs))


In [ ]:
# normalize both sides for matching
hard_set = set(zip(hard["col1"], hard["col2"]))

# check each pair in both directions
pairs["hit"] = [
    (a, b) in hard_set or (b, a) in hard_set
    for a, b in zip(pairs["code_a"], pairs["code_b"])
]

violations = pairs[pairs["hit"]]
print("Violating pairs:", len(violations))
print("Claims with at least one violation:", violations["CLM_ID"].nunique())

In [ ]:
v_claims = set(violations["CLM_ID"])
v_dollars = claim_totals[claim_totals["CLM_ID"].isin(v_claims)]["CLM_PMT_AMT"].sum()

print("Claims with hard-edit violations:", len(v_claims))
print("Dollars on those claims: $", v_dollars)
print("Share of total dollars:", round(100 * v_dollars / claim_totals["CLM_PMT_AMT"].sum(), 1), "%")

In [ ]:
top = (violations.groupby(["code_a", "code_b"])
       .size()
       .sort_values(ascending=False)
       .head(15))
print(top.to_string())

In [ ]:
violations.to_csv("violations.csv", index=False)
top.to_csv("top_violation_pairs.csv")

In [ ]:
violations["CLM_ID"] = violations["CLM_ID"].astype("int64")

clm_prov = claims[["CLM_ID", "PRVDR_NUM"]].drop_duplicates("CLM_ID")

v_claims = violations["CLM_ID"].unique()
prov_viol = clm_prov[clm_prov["CLM_ID"].isin(v_claims)]

print("Violating claims mapped:", len(prov_viol))
print("Distinct providers with violations:", prov_viol["PRVDR_NUM"].nunique())

In [ ]:
top_prov = prov_viol["PRVDR_NUM"].value_counts().head(20)
print(top_prov.to_string())

total_prov = clm_prov["PRVDR_NUM"].nunique()
counts = prov_viol["PRVDR_NUM"].value_counts()
top10_share = counts.head(10).sum() / counts.sum()
top50_share = counts.head(50).sum() / counts.sum()

print("\nTotal providers in dataset:", total_prov)
print("Providers with violations:", len(counts))
print("Share of violations from top 10 providers:", round(100 * top10_share, 1), "%")
print("Share of violations from top 50 providers:", round(100 * top50_share, 1), "%")

In [ ]:
# how many service lines does each violating claim have?
lines_per_claim = lines.groupby("CLM_ID").size().rename("n_lines")

v_detail = (claim_totals[claim_totals["CLM_ID"].isin(v_claims)]
            .merge(lines_per_claim, on="CLM_ID"))

# violating pairs per claim
pairs_per_claim = violations.groupby("CLM_ID").size().rename("n_viol")
v_detail = v_detail.merge(pairs_per_claim, on="CLM_ID")

# estimate: one denied line per violating pair, valued at claim's average line value
v_detail["est_line_value"] = v_detail["CLM_PMT_AMT"] / v_detail["n_lines"]
v_detail["est_exposure"] = v_detail["est_line_value"] * v_detail["n_viol"]

print("Claims touched (upper bound): $", round(v_detail["CLM_PMT_AMT"].sum(), 2))
print("Estimated line-level exposure: $", round(v_detail["est_exposure"].sum(), 2))
print("Median lines per violating claim:", v_detail["n_lines"].median())